# MLPerf Inference — ResNet-50 / ImageNet (Vision, local WSL)

Vision classification on your RTX 5070 Ti. Default dataset = **Imagenette** (ungated,
10 real-ImageNet classes → top-1 ≈ 84.5%). Optional cell 8 swaps in a **representative
1000-class ImageNet-1k subset** (≈ 76%) from an ungated HF mirror.

> **How to run (in the `mlperf` WSL distro):**
> ```bash
> wsl -d mlperf
> source /root/mlperf/venv/bin/activate
> pip install -q jupyterlab ipykernel
> python -m ipykernel install --user --name mlperf --display-name 'mlperf (venv)'
> jupyter lab --no-browser --ip 0.0.0.0 --port 8888   # open the printed URL
> ```
> Then open this notebook, pick the **mlperf (venv)** kernel, and Run All.
> Assets are cached under `/root/mlperf/…`, so re-runs skip downloads. Works on a fresh
> distro too (it clones + downloads on first run). Needs torch 2.11+cu128 already in the venv.

## 1 · GPU check

In [1]:
import torch; print('cuda',torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

cuda True NVIDIA GeForce RTX 5070 Ti Laptop GPU


## 2 · Deps

In [ ]:
%pip install -q "mlcommons-loadgen==6.0.16" "pycocotools==2.0.11" "opencv-python-headless==5.0.0.93"
%pip install -q "torchvision==0.26.0" --index-url https://download.pytorch.org/whl/cu128
import torchvision, cv2, pycocotools; print('torchvision',torchvision.__version__,'cv2',cv2.__version__)

## 3 · Repo + model + Imagenette

In [ ]:
%%bash
set -euo pipefail
INFERENCE_REF=da738a5     # pin the harness (mlcommons/inference is a moving master)
verify(){ echo "$1  $2" | sha256sum -c - >/dev/null || { echo "!! checksum mismatch: $2"; exit 1; }; }

if [ ! -d /root/mlperf/inference ]; then
  git clone --filter=blob:none --no-checkout https://github.com/mlcommons/inference.git /root/mlperf/inference
fi
git -C /root/mlperf/inference fetch --filter=blob:none -q origin "$INFERENCE_REF" 2>/dev/null || true
W=$(git -C /root/mlperf/inference rev-parse -q --verify "$INFERENCE_REF^{commit}")
git -C /root/mlperf/inference reset --hard -q "$W"    # pin exactly, every run

mkdir -p /root/mlperf/vision && cd /root/mlperf/vision
[ -s resnet50-19c8e357.pth ] || wget -q -O resnet50-19c8e357.pth 'https://zenodo.org/record/4588417/files/resnet50-19c8e357.pth?download=1'
verify 19c8e3572231adff6824a2da93fd67b5986919a2e65f8b6007eab4edee220097 resnet50-19c8e357.pth

# Imagenette: hash-enforced by DEFAULT against the canonical fast.ai archive (verify BEFORE extract).
# The HASH is the trust anchor, so any mirror matching it is fine; override IMAGENETTE_SHA256 to pin a
# different snapshot. See reference/CHECKSUMS.md.
IMAGENETTE_SHA256="${IMAGENETTE_SHA256:-569b4497c98db6dd29f335d1f109cf315fe127053cedf69010d047f0188e158c}"
if [ ! -d imagenette2-320 ]; then
  wget -q -O i.tgz https://s3.amazonaws.com/fast-ai-imageclas/imagenette2-320.tgz
  verify "$IMAGENETTE_SHA256" i.tgz    # aborts on a bad/partial download or a wrong mirror
  tar xzf i.tgz && rm i.tgz
fi
echo 'val images:' $(find imagenette2-320/val -iname '*.JPEG' | wc -l)

## 4 · Full model + val_map + backend patch

Legacy-format state_dict → full model (`weights_only=False`); patch the pytorch-native
backend to return a numpy batch; build `val_map.txt` with the 10 synset→index labels.

In [ ]:
import torch, torchvision, os, glob
# resnet50-19c8e357.pth is SHA-256-verified in cell 3, so this checkpoint is trusted. It's a legacy
# tar-format state_dict that needs weights_only=False; try the safe path first and fall back.
PTH='/root/mlperf/vision/resnet50-19c8e357.pth'
try:
    sd=torch.load(PTH, map_location='cpu', weights_only=True)
except Exception:
    sd=torch.load(PTH, map_location='cpu', weights_only=False)   # verified file -> trusted pickle
m=torchvision.models.resnet50(); m.load_state_dict(sd); m.eval()
torch.save(m,'/root/mlperf/vision/resnet50_full.pth'); print('saved full model')
SYN={'n01440764':0,'n02102040':217,'n02979186':482,'n03000684':491,'n03028079':497,
     'n03394916':566,'n03417042':569,'n03425413':571,'n03445777':574,'n03888257':701}
vd='/root/mlperf/vision/imagenette2-320/val'; L=[]
for s,i in SYN.items():
    for p in sorted(glob.glob(os.path.join(vd,s,'*.JPEG'))): L.append(f'{os.path.relpath(p,vd)} {i}')
open(os.path.join(vd,'val_map.txt'),'w').write('\n'.join(L)+'\n'); print('val_map:',len(L))
# Patch backend_pytorch_native to return [numpy batch]. FAIL LOUD if the anchor is missing (a bumped
# INFERENCE_REF / upstream drift) instead of silently no-op'ing and lying 'already patched' — an
# unpatched backend returns a raw tensor and breaks the accuracy scorer downstream.
f='/root/mlperf/inference/vision/classification_and_detection/python/backend_pytorch_native.py'
s=open(f).read()
old='        with torch.no_grad():\n            output = self.model(feed[key])\n        return output'
new='        with torch.no_grad():\n            output = self.model(feed[key])\n        return [output.cpu().numpy()]'
if new in s:
    print('backend already patched')
elif old in s:
    s2=s.replace(old, new, 1); assert new in s2, 'replace() did not insert the numpy return'
    open(f, 'w').write(s2); print('backend patched')
else:
    raise SystemExit('anchor not found in backend_pytorch_native.py — upstream layout changed; '
                     'update the patch (do not run with an unpatched backend)')

## 5 · Accuracy (top-1) + Performance

main.py accuracy mode crashes in post-test stats after writing the log → tolerate & score externally.

In [5]:
%%bash
source /root/mlperf/venv/bin/activate
cd /root/mlperf/inference/vision/classification_and_detection
cat > /root/mlperf/vision/demo.conf <<'CONF'
resnet50.Offline.target_qps = 1000
resnet50.Offline.min_duration = 10000
resnet50.Offline.min_query_count = 1
CONF
cd python
python main.py --profile resnet50-pytorch --backend pytorch-native --model /root/mlperf/vision/resnet50_full.pth \
  --dataset-path /root/mlperf/vision/imagenette2-320/val --user_conf /root/mlperf/vision/demo.conf --scenario Offline --accuracy \
  --output /root/mlperf/vision/out_acc >/tmp/a.log 2>&1 || true
echo '----- top-1 -----'; python ../tools/accuracy-imagenet.py --mlperf-accuracy-file /root/mlperf/vision/out_acc/mlperf_log_accuracy.json --imagenet-val-file /root/mlperf/vision/imagenette2-320/val/val_map.txt --dtype float32 2>&1 | tail -1
echo '----- perf -----'
python main.py --profile resnet50-pytorch --backend pytorch-native --model /root/mlperf/vision/resnet50_full.pth \
  --dataset-path /root/mlperf/vision/imagenette2-320/val --user_conf /root/mlperf/vision/demo.conf --scenario Offline \
  --output /root/mlperf/vision/out_perf >/tmp/p.log 2>&1 || true
sed -n '1,11p' /root/mlperf/vision/out_perf/mlperf_log_summary.txt

----- top-1 -----
accuracy=84.510%, good=3317, total=3925
----- perf -----
MLPerf Results Summary
SUT name : PySUT
Scenario : Offline
Mode     : PerformanceOnly
Samples per second: 472.408
Result is : VALID
  Min duration satisfied : Yes
  Min queries satisfied : Yes
  Early stopping satisfied: Yes


## Done ✅ — ResNet-50 top-1 (Imagenette) + VALID perf.

---
## Optional · Representative ImageNet-1k (all 1000 classes → ≈76%)

Downloads the **ungated** `Tsomaros/Imagenet-1k_validation` (6.7 GB, full 50k val, standard
1000-class labels), keeps 5/class = 5000 imgs, and re-scores. No HF token needed. Then re-run
cell 6-style scoring pointing at `/root/mlperf/vision/inet_val`.

In [6]:
# Set True to build the representative subset (downloads 6.7 GB the first time).
# Left False so 'Run All' stays light — flip it and run the next two cells manually.
RUN_REPRESENTATIVE = False

In [ ]:
%pip install -q "huggingface_hub>=0.24,<1.0" "pyarrow==25.0.0"

In [8]:
import os
if not RUN_REPRESENTATIVE:
    print('skipped (set RUN_REPRESENTATIVE=True to build the 6.7 GB subset)')
else:
    import pyarrow.parquet as pq
    from huggingface_hub import HfApi, hf_hub_download
    REPO_ID='Tsomaros/Imagenet-1k_validation'
    files=sorted(f for f in HfApi().list_repo_files(REPO_ID, repo_type='dataset') if f.endswith('.parquet'))
    OUT='/root/mlperf/vision/inet_val'; os.makedirs(OUT, exist_ok=True)
    g=0; k=0; vm=open(os.path.join(OUT,'val_map.txt'),'w')
    for fn in files:
        t=pq.read_table(hf_hub_download(REPO_ID, fn, repo_type='dataset'))
        for img,lab in zip(t.column('image').to_pylist(), t.column('label').to_pylist()):
            if g%10==0:
                b=img['bytes'] if isinstance(img,dict) else img
                open(os.path.join(OUT,f'{k:05d}.JPEG'),'wb').write(b); vm.write(f'{k:05d}.JPEG {int(lab)}\n'); k+=1
            g+=1
        print(fn,'seen',g,'kept',k)
    vm.close(); print('TOTAL', k)

skipped (set RUN_REPRESENTATIVE=True to build the 6.7 GB subset)


In [9]:
%%bash
source /root/mlperf/venv/bin/activate
cd /root/mlperf/inference/vision/classification_and_detection/python
if [ ! -s /root/mlperf/vision/inet_val/val_map.txt ]; then echo 'representative subset not built — set RUN_REPRESENTATIVE=True and run the cell above'; exit 0; fi
python main.py --profile resnet50-pytorch --backend pytorch-native --model /root/mlperf/vision/resnet50_full.pth \
  --dataset-path /root/mlperf/vision/inet_val --user_conf /root/mlperf/vision/demo.conf --scenario Offline --accuracy \
  --output /root/mlperf/vision/inet_out_acc >/tmp/ia.log 2>&1 || true
echo '----- representative top-1 (1000 classes) -----'
python ../tools/accuracy-imagenet.py --mlperf-accuracy-file /root/mlperf/vision/inet_out_acc/mlperf_log_accuracy.json --imagenet-val-file /root/mlperf/vision/inet_val/val_map.txt --dtype float32 2>&1 | tail -1

----- representative top-1 (1000 classes) -----
accuracy=75.400%, good=3770, total=5000
